In [ ]:
#instalando dependências
!pip install experta
print('Uninstalling old frozendict and installing a compatible version...')
!pip install --upgrade frozendict

  Using cached frozendict-1.2-py3-none-any.whl
  Attempting uninstall: frozendict
    Found existing installation: frozendict 2.4.7
    Uninstalling frozendict-2.4.7:
      Successfully uninstalled frozendict-2.4.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
Uninstalling old frozendict and installing a compatible version...
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
Using cached frozendict-2.4.7-py3-none-any.whl (16 kB)
  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency c

In [ ]:
#importações necessárias
from experta import KnowledgeEngine, Rule, Fact, AND, P, OR, NOT

In [ ]:


#declaração das estruturas de dados (Fatos)

class Sintoma(Fact):
    """
    Campos esperados:
    - km_oleo (int): quilometragem desde a última troca
    - folga_corrente_cm (float): medida da folga em centímetros
    - som_partida (str): 'normal' ou 'cliques'
    - freio (str): 'normal' ou 'borrachudo'
    - barulho_motor (str): 'normal' ou 'metalico'
    - marcha (str): 'macia' ou 'dura'
    - vazamento (bool): True ou False
    """
    pass

class EstadoComponente(Fact):
    """
    Campos esperados:
    - oleo (str): 'vencido' ou 'bom'
    - corrente (str): 'frouxa' ou 'ok'
    - bateria (str): 'fraca' ou 'ok'
    """
    pass

class Diagnostico(Fact):
    """
    Campos esperados:
    - problema (str): 'desgaste_acelerado', 'troca_rotina', 'kit_relacao_ressecado', etc.
    """
    pass

class AcaoRecomendada(Fact):
    """
    Campos esperados:
    - acao (str): texto descritivo do que deve ser feito
    - urgencia (str): 'baixa' ou 'maxima'
    """
    pass

In [ ]:
class MotorDiagnosticoMoto(KnowledgeEngine):

  #nível 1: Sintoma -> Estado do Componente

  #R1
  @Rule(Sintoma(km_oleo=P(lambda k: k >= 2000)))
  def estado_oleo_vencido(self):
    print("[LOG] Regra 1 ativada: Óleo vencido detectado (>= 2000km).")
    self.declare(EstadoComponente(oleo="vencido"))
  #R2
  @Rule(Sintoma(km_oleo=P(lambda k: k < 2000)))
  def estado_oleo_bom(self):
    print("[LOG] Regra 2 ativada: Óleo em bom estado (< 2000km).")
    self.declare(EstadoComponente(oleo="bom"))
  #R3
  @Rule(Sintoma(folga_corrente_cm=P(lambda c: c >= 3.0)))
  def estado_corrente_frouxa(self):
    print("[LOG] Regra 3 ativada: Corrente frouxa detectada (>= 3.0cm).")
    self.declare(EstadoComponente(corrente="frouxa"))
  #R4
  @Rule(Sintoma(som_partida="cliques"))
  def estado_bateria_fraca(self):
    print("[LOG] Regra 4 ativada: Bateria fraca detectada (som de cliques).")
    self.declare(EstadoComponente(bateria="fraca"))
  #R5
  @Rule(Sintoma(freio="borrachudo"), salience=100)
  def alerta_freio(self):
    print("[LOG] Regra 5 ativada (Salience=100): Anomalia crítica nos freios!")
    print("-> AÇÃO [MAXIMA]: Alerta! Freio borrachudo. Trocar o freio imediatamente.")
    print("   (Trace: Decisão tomada isoladamente pela Regra 5 devido ao risco crítico, atropelando o encadeamento padrão)")
    self.declare(AcaoRecomendada(acao="Trocar o freio", urgencia="MAXIMA"))


  #nível 2: Estado -> Diagnóstico

  #R6
  @Rule(EstadoComponente(oleo="vencido"), Sintoma(barulho_motor="metalico"))
  def diag_desgaste_motor(self):
    print("[LOG] Regra 6 ativada: Cruzou óleo vencido com barulho metálico. Diagnóstico: Desgaste.")
    self.declare(Diagnostico(problema="desgaste_acelerado"))

  #R7
  @Rule(EstadoComponente(oleo="vencido"), NOT(Sintoma(barulho_motor="metalico")))
  def diag_troca_rotina(self):
    print("[LOG] Regra 7 ativada: Cruzou óleo vencido sem barulho (NOT). Diagnóstico: Troca de Rotina.")
    self.declare(Diagnostico(problema="troca_rotina"))

  #R8
  @Rule(EstadoComponente(corrente="frouxa"), Sintoma(marcha="dura"))
  def diag_kit_relacao(self):
    print("[LOG] Regra 8 ativada: Cruzou corrente frouxa com marcha dura. Diagnóstico: Kit Ressecado.")
    self.declare(Diagnostico(problema="kit_relacao_ressecado"))

  #R9
  @Rule(EstadoComponente(bateria="fraca"))
  def diag_bateria(self):
    print("[LOG] Regra 9 ativada: Bateria fraca confirmada. Diagnóstico: Sem Carga.")
    self.declare(Diagnostico(problema="bateria_sem_carga"))

  #nível 3: Diagnóstico -> Ação Recomendada

  #R10
  @Rule(Diagnostico(problema="desgaste_acelerado"), Sintoma(vazamento=True))
  def acao_risco_fundir(self):
    print("[LOG] Regra 10 ativada: Desgaste + Vazamento confirmados.")
    print("\n-> AÇÃO FINAL: Chamar guincho. Motor com desgaste, barulho e vazamento. Risco altíssimo de fundir.")
    print("   (Trace Final: Decisão tomada porque as Regras 1, 6 e 10 dispararam em cadeia)\n")

  #R11
  @Rule(Diagnostico(problema="desgaste_acelerado"), NOT(Sintoma(vazamento=True)))
  def acao_verificar_cilindro(self):
    print("[LOG] Regra 11 ativada: Desgaste confirmado (Sem vazamento).")
    print("\n-> AÇÃO FINAL: Levar ao mecânico devagar. Trocar óleo e verificar cilindro do motor devido ao barulho metálico.")
    print("   (Trace Final: Decisão tomada porque as Regras 1, 6 e 11 dispararam em cadeia)\n")

  #R12
  @Rule(Diagnostico(problema="troca_rotina"))
  def acao_trocar_oleo(self):
    print("[LOG] Regra 12 ativada: Troca de Rotina confirmada.")
    print("\n-> AÇÃO FINAL: Realizar manutenção preventiva: Troca de óleo 10W-30 semissintético.")
    print("   (Trace Final: Decisão tomada porque as Regras 1, 7 e 12 dispararam em cadeia)\n")

  #R13
  @Rule(Diagnostico(problema="kit_relacao_ressecado"))
  def acao_ajustar_corrente(self):
    print("[LOG] Regra 13 ativada: Kit Ressecado confirmado.")
    print("\n-> AÇÃO FINAL: Esticar corrente de transmissão e aplicar óleo lubrificante espesso.")
    print("   (Trace Final: Decisão tomada porque as Regras 3, 8 e 13 dispararam em cadeia)\n")

In [ ]:

#caso de teste 1: Desgaste Acelerado (Sem Vazamento) e Kit Relação
#saída Esperada:
#o sistema deve recomendar duas ações: "Levar ao mecânico devagar"
#(para verificar o cilindro) e "Esticar corrente e aplicar óleo".
#explicação:
#o óleo está com 2100km, ativando a Regra 1 (óleo vencido). O sistema
#cruza isso com o barulho "metálico", gerando o diagnóstico de desgaste
#na Regra 6. Como o vazamento é 'False', a cláusula NOT da Regra 11 é
#ativada, recomendando levar ao mecânico com cautela.
#simultaneamente, a folga de 3.5cm ativa a Regra 3, a marcha "dura"
#ativa a Regra 8, e a ação final de ajuste da corrente é recomendad pela Regra 13.
engine = MotorDiagnosticoMoto()
engine.reset()

engine.declare(Sintoma(
    km_oleo=2100,
    barulho_motor="metalico",
    folga_corrente_cm=3.5,
    marcha="dura",
    vazamento=False,
    freio="normal",
    som_partida="normal"
))

print("iniciando inferência do sistema...\n")
engine.run()

iniciando inferência do sistema...

[LOG] Regra 3 ativada: Corrente frouxa detectada (>= 3.0cm).
[LOG] Regra 8 ativada: Cruzou corrente frouxa com marcha dura. Diagnóstico: Kit Ressecado.
[LOG] Regra 13 ativada: Kit Ressecado confirmado.

-> AÇÃO FINAL: Esticar corrente de transmissão e aplicar óleo lubrificante espesso.
   (Trace Final: Decisão tomada porque as Regras 3, 8 e 13 dispararam em cadeia)

[LOG] Regra 1 ativada: Óleo vencido detectado (>= 2000km).
[LOG] Regra 6 ativada: Cruzou óleo vencido com barulho metálico. Diagnóstico: Desgaste.
[LOG] Regra 11 ativada: Desgaste confirmado (Sem vazamento).

-> AÇÃO FINAL: Levar ao mecânico devagar. Trocar óleo e verificar cilindro do motor devido ao barulho metálico.
   (Trace Final: Decisão tomada porque as Regras 1, 6 e 11 dispararam em cadeia)



In [ ]:
#caso de teste 2: múltiplos problemas (guincho + corrente)
#Saída Esperada:
#O sistema deve recomendar duas ações: "chamar guincho" e "esticar corrente".
#explicação: o óleo está vencido (Regra 1) e há barulho metálico, gerando o
#diagnóstico de desgaste (Regra 6). O vazamento ativa a Regra 10 (guincho).
#simultaneamente, a folga de 4.5cm ativa a Regra 3, a marcha dura ativa a regra 8, e a ação final para a corrente é recomendada pela Regra 13.

engine = MotorDiagnosticoMoto()
engine.reset()

engine.declare(Sintoma(
    km_oleo=2500,
    barulho_motor="metalico",
    folga_corrente_cm=4.5,
    marcha="dura",
    vazamento=True,
    freio="normal",
    som_partida="normal"
))

print("INICIANDO TESTE 2")
engine.run()

INICIANDO TESTE 2
[LOG] Regra 3 ativada: Corrente frouxa detectada (>= 3.0cm).
[LOG] Regra 8 ativada: Cruzou corrente frouxa com marcha dura. Diagnóstico: Kit Ressecado.
[LOG] Regra 13 ativada: Kit Ressecado confirmado.

-> AÇÃO FINAL: Esticar corrente de transmissão e aplicar óleo lubrificante espesso.
   (Trace Final: Decisão tomada porque as Regras 3, 8 e 13 dispararam em cadeia)

[LOG] Regra 1 ativada: Óleo vencido detectado (>= 2000km).
[LOG] Regra 6 ativada: Cruzou óleo vencido com barulho metálico. Diagnóstico: Desgaste.
[LOG] Regra 10 ativada: Desgaste + Vazamento confirmados.

-> AÇÃO FINAL: Chamar guincho. Motor com desgaste, barulho e vazamento. Risco altíssimo de fundir.
   (Trace Final: Decisão tomada porque as Regras 1, 6 e 10 dispararam em cadeia)



In [ ]:
#caso teste 3: Resolução de Conflito (Salience = 100)
#saída Esperada:
#o sistema deve ignorar o encadeamento padrão e exibir imediatamente o
#aviso crítico do freio borrachudo.
#explicação: embora a bateria esteja com cliques (regra 4) e devesse acionar a regra 9,
#a regra do freio (regra 5) possui 'salience=100'.
#O motor de inferência resolve o conflito executando a Regra 5 primeiro devido ao risco de vida.

engine = MotorDiagnosticoMoto()
engine.reset()

engine.declare(Sintoma(
    km_oleo=1500,
    barulho_motor="normal",
    folga_corrente_cm=1.0,
    marcha="macia",
    vazamento=False,
    freio="borrachudo", # gatilho da resolução de conflito
    som_partida="cliques"
))

print("INICIANDO TESTE 3")
engine.run()

INICIANDO TESTE 3
[LOG] Regra 5 ativada (Salience=100): Anomalia crítica nos freios!
-> AÇÃO [MAXIMA]: Alerta! Freio borrachudo. Trocar o freio imediatamente.
   (Trace: Decisão tomada isoladamente pela Regra 5 devido ao risco crítico, atropelando o encadeamento padrão)
[LOG] Regra 4 ativada: Bateria fraca detectada (som de cliques).
[LOG] Regra 9 ativada: Bateria fraca confirmada. Diagnóstico: Sem Carga.
[LOG] Regra 2 ativada: Óleo em bom estado (< 2000km).
